# SUPPORT2 Dataset: Model Selection
**AAI-500 Applied Statistics for AI | Final Team Project — Group 3**

*Evaluating Early Clinical Indicators for 180-Day Mortality: A Comparative Analysis Using the SUPPORT2 Dataset*

---

## Notebook Objectives

This notebook covers the **Model Selection** phase of the project pipeline:

1. Load model-ready feature set and apply feature exclusions
2. Preprocessing: scale numeric features, encode categoricals
3. Baseline logistic regression (no regularization)
4. LASSO-regularized logistic regression (L1 penalty, tuned)
5. Random Forest classifier (tuned)
6. ROC curve comparison: all models vs. SUPPORT benchmark (AUC = 0.787)
7. Model selection and justification

**Benchmark:** Knaus et al. (1995) reported an AUC of 0.78 for the SUPPORT
prognostic model on independent phase II validation data. Our EDA confirmed
this holds in the current dataset (AUC = 0.787 using `surv6m`). Matching or
exceeding this AUC is the primary performance target.

**Reference:** Knaus, W. A., Harrell, F. E., Lynn, J., Goldman, L., Phillips,
R. S., Connors, A. F., ... & Wagner, D. P. (1995). The SUPPORT prognostic
model: Objective estimates of survival for seriously ill hospitalized adults.
*Annals of Internal Medicine, 122*(3), 191–203.

---

## Section 0: Imports

In [ ]:
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

sys.path.append("../../utils")
from dataset import load_csv

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
SUPPORT_AUC = 0.787  # Knaus et al. (1995), confirmed in EDA notebook Section 9

---
## Section 1: Load Data & Feature Selection

In [ ]:
df = load_csv("support2_model_features.csv")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

### 1.1 Feature Exclusions

The following columns are excluded from the feature set on methodological grounds.

**Leakage: prognostic system outputs (`sps`, `surv6m`, `surv2m`, `prg6m`, `prg2m`, `aps`)**

These columns are outputs of competing prognostic systems, not raw clinical inputs.
Including them would would test our models' ability to replicate existing scoring systems rather than predict mortality from clinical
data.

- `sps`: SUPPORT physiology sub-score, a direct component of the SUPPORT model
  whose output (`surv6m`) is our benchmark. Including it would be circular.
- `surv6m`, `surv2m`: SUPPORT model survival probability outputs. Retained in
  `support2_cleaned.csv` for benchmark comparison; excluded here because they
  ARE the benchmark.
- `prg6m`, `prg2m`: physician survival estimates collected at study day 3.
  Including them replicates the physician-enhanced SUPPORT model rather than
  an independent classifier.
- `aps`: APACHE III composite severity score. Knaus et al. (1995) evaluated
  SUPPORT and APACHE III as parallel competing systems (Table 2); neither used
  the other's score as an input feature. Including `aps` has no methodological
  precedent in the source paper.

**Administrative cost variables (`charges`, `totcst`, `totmcst`)**

Excluded on two grounds:

1. **Temporal leakage**: costs accumulate during the hospitalization as a
   consequence of care delivered. They reflect illness trajectory and intervention
   intensity rather than predicting mortality independently. The variable is
   downstream of the outcome, not upstream of it.
2. **Redundancy**: `totcst` and `charges` are highly correlated (r = 0.742,
   EDA Section 12), carrying no independent signal from one another.

**Non-clinical enrollment variable (`hday`)**

`hday` records the day the patient entered the study. It reflects enrollment
timing, not clinical status at day 3, and has no causal pathway to 180-day
mortality.

In [ ]:
TARGET = "death_180d"  # exclude the target variable itself
LEAKAGE_COLS = ["sps", "surv6m", "surv2m", "prg6m", "prg2m", "aps"]
ADMIN_COLS = ["charges", "totcst", "totmcst"]
NON_CLINICAL_COLS = ["hday"]

feature_cols = [
    c
    for c in df.columns
    if c != TARGET
    and c not in LEAKAGE_COLS
    and c not in ADMIN_COLS
    and c not in NON_CLINICAL_COLS
]

X = df[feature_cols]
y = df[TARGET]

print(f"Features: {len(feature_cols)}")
print(f"Target: {TARGET}")
print("\nClass balance:")
print(
    y.value_counts(normalize=True)
    .rename({0: "Survived", 1: "Died"})
    .rename_axis(None)
    .to_string()
)

### 1.2 Train/Test Split

In [ ]:
# Stratified split preserves class balance across train and test sets.
# 80/20 split gives sufficient training data while holding out a meaningful
# evaluation set. random_state ensures reproducibility across runs.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape[0]:,} rows")
print(f"Test: {X_test.shape[0]:,} rows")
print(f"\nTrain class balance: {y_train.mean():.4f} positive rate")
print(f"Test class balance: {y_test.mean():.4f} positive rate")

---
## Section 2: Preprocessing

In [ ]:
# Identify column types for preprocessing routing.
# Nominal categoricals get one-hot encoding (no ordinal assumption).
# Ordinal categoricals get ordinal encoding (preserves rank order from cleaning notebook).
# All numeric features get standard scaling (required for logistic regression / LASSO).

NOMINAL_CATS = ["sex", "dzgroup", "dzclass", "race", "ca", "dnr"]
ORDINAL_CATS = ["income", "sfdm2"]

# Ordinal category orders established in [EDA REF]
INCOME_ORDER = ["under $11k", "$11-$25k", "$25-$50k", ">$50k"]
SFDM2_ORDER = [
    "no(M2 and SIP pres)",
    "adl>=4 (>=5 if sur)",
    "SIP>=30",
    "Coma or Intub",
    "<2 mo. follow-up",
]

# Filter to only columns present in X after leakage exclusion
nominal_present = [c for c in NOMINAL_CATS if c in feature_cols]
ordinal_present = [c for c in ORDINAL_CATS if c in feature_cols]
numeric_present = [c for c in feature_cols if c not in NOMINAL_CATS + ORDINAL_CATS]

print(f"Numeric features: {len(numeric_present)}  {numeric_present}")
print(f"Nominal features: {len(nominal_present)}  {nominal_present}")
print(f"Ordinal features: {len(ordinal_present)}  {ordinal_present}")

In [ ]:
# ColumnTransformer applies different preprocessing to each column type in one step.
# drop='first' on OneHotEncoder avoids the dummy variable trap in logistic regression.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_present),
        (
            "nom",
            OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False),
            nominal_present,
        ),
        (
            "ord",
            OrdinalEncoder(
                categories=[INCOME_ORDER, SFDM2_ORDER],
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
            ordinal_present,
        ),
    ],
    remainder="drop",
)

print("Preprocessor built successfully.")

---
## Section 3: Baseline Logistic Regression

Logistic regression with no regularization establishes a performance floor.
It is interpretable, fast, and directly comparable to the Cox proportional
hazards model used by Knaus et al. (1995), which shares the same log-odds
foundation. A weak baseline result motivates regularization or ensemble methods;
a strong baseline suggests the outcome is largely linearly separable in feature space.

In [ ]:
pipe_lr = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                penalty=None,  # no regularization — true baseline
                solver="lbfgs",
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

pipe_lr.fit(X_train, y_train)

y_pred_lr = pipe_lr.predict(X_test)
y_prob_lr = pipe_lr.predict_proba(X_test)[:, 1]

metrics_lr = {
    "Model": "Logistic Regression (baseline)",
    "Accuracy": accuracy_score(y_test, y_pred_lr),
    "Precision": precision_score(y_test, y_pred_lr),
    "Recall": recall_score(y_test, y_pred_lr),
    "F1": f1_score(y_test, y_pred_lr),
    "ROC AUC": roc_auc_score(y_test, y_prob_lr),
}

print("Baseline Logistic Regression: Test Set Performance")
for k, v in metrics_lr.items():
    if k != "Model":
        print(f"  {k:<12} {v:.4f}")
print(f"\n  SUPPORT benchmark AUC: {SUPPORT_AUC:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_lr),
    display_labels=["Survived", "Died"],
).plot(ax=ax, colorbar=False)
ax.set_title("Confusion Matrix: Baseline Logistic Regression", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Section 4: LASSO Logistic Regression

L1 (LASSO) regularization shrinks less informative coefficients to exactly zero,
performing implicit feature selection. This is appropriate here because:

1. Seven features were non-significant in EDA hypothesis testing (temp, num_co,
   bun, urine, ph, sod, glucose); LASSO can downweight these without manual exclusion
2. Correlated feature pairs (aps × sps, already excluded) can cause coefficient
   instability in unregularized logistic regression
3. The resulting sparse model is more interpretable for a clinical audience

Note: reference categories for nominal features are determined by alphabetical 
ordering via scikit-learn's `drop='first'` parameter. Coefficient values are 
interpretable only relative to those implicit baselines and are reported here 
as supporting evidence rather than primary findings.

Regularization strength C (inverse of lambda) is tuned via 5-fold cross-validation.

### 4.1 Hyperparameter Tuning

In [ ]:
pipe_lasso = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                penalty="l1",
                solver="liblinear",  # liblinear is required for L1 penalty
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

# C is inverse regularization strength: smaller C = stronger penalty.
# Log-spaced search (10^-3 to 10^2) across 20 values covers a wide range
# without exhaustive computation.
param_grid_lasso = {"classifier__C": np.logspace(-3, 2, 20)}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid_lasso = GridSearchCV(
    pipe_lasso,
    param_grid_lasso,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=0,
)
grid_lasso.fit(X_train, y_train)

best_C_lasso = grid_lasso.best_params_["classifier__C"]
print(f"Best C (LASSO): {best_C_lasso:.4f}")
print(f"Best CV AUC (LASSO): {grid_lasso.best_score_:.4f}")

### 4.2 Test Set Evaluation

In [ ]:
best_lasso = grid_lasso.best_estimator_

y_pred_lasso = best_lasso.predict(X_test)
y_prob_lasso = best_lasso.predict_proba(X_test)[:, 1]

metrics_lasso = {
    "Model": "LASSO Logistic Regression",
    "Accuracy": accuracy_score(y_test, y_pred_lasso),
    "Precision": precision_score(y_test, y_pred_lasso),
    "Recall": recall_score(y_test, y_pred_lasso),
    "F1": f1_score(y_test, y_pred_lasso),
    "ROC AUC": roc_auc_score(y_test, y_prob_lasso),
}

print("LASSO Logistic Regression — Test Set Performance")
for k, v in metrics_lasso.items():
    if k != "Model":
        print(f"  {k:<12} {v:.4f}")
print(f"\n  SUPPORT benchmark AUC: {SUPPORT_AUC:.4f}")

### 4.3 Coefficient Analysis

In [ ]:
# Inspect which features LASSO zeroed out at the best C.
# Zero coefficients = features LASSO deemed uninformative given the regularization budget.
lasso_clf = best_lasso.named_steps["classifier"]
preprocessed_feature_names = (
    best_lasso.named_steps["preprocessor"].get_feature_names_out().tolist()
)

coef_df = pd.DataFrame(
    {"Feature": preprocessed_feature_names, "Coefficient": lasso_clf.coef_[0]}
).sort_values("Coefficient", key=abs, ascending=False)

n_zeroed = (coef_df["Coefficient"] == 0).sum()
print(f"Features zeroed by LASSO : {n_zeroed} / {len(coef_df)}")
print("\nTop 15 features by absolute coefficient:")
print(coef_df.head(15).to_string(index=False, float_format="{:.4f}".format))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_lasso),
    display_labels=["Survived", "Died"],
).plot(ax=ax, colorbar=False)
ax.set_title("Confusion Matrix — LASSO Logistic Regression", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Section 5: Random Forest

Random Forest is an ensemble of decision trees trained on bootstrap samples
with random feature subsets at each split. It captures nonlinear relationships
and feature interactions, including the albumin × disease group and
age × disease group interactions identified in [EDA REF], without
requiring explicit interaction terms to be specified.

The cost is reduced interpretability relative to logistic regression.
Feature importance scores (mean decrease in impurity) partially recover
interpretability by ranking predictors by contribution across all trees.

The EDA phase directly addresses clinical factor identification through
hypothesis testing and interaction analysis [EDA REF] so the modeling phase
is free to prioritize predictive performance over interpretability; this
makes Random Forest a justified candidate despite this tradeoff.

### 5.1 Learning Curve: Number of Trees

Before tuning hyperparameters, we establish an empirical basis for selecting
the number of trees (`n_estimators`). The learning curve plots mean cross-validated
AUC against a range of tree counts, identifying where performance stabilizes and
additional trees yield diminishing returns.

In [ ]:
from sklearn.model_selection import cross_val_score

n_estimator_range = [50, 100, 200, 300, 500, 750, 1000]
mean_aucs = []

for n in n_estimator_range:
    rf = Pipeline(
        [
            ("preprocessor", preprocessor),
            (
                "classifier",
                RandomForestClassifier(
                    n_estimators=n, random_state=RANDOM_STATE, n_jobs=-1
                ),
            ),
        ]
    )
    scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring="roc_auc")
    mean_aucs.append(scores.mean())

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(n_estimator_range, mean_aucs, marker="o", linewidth=2)
ax.set_xlabel("n_estimators")
ax.set_ylabel("Mean CV AUC")
ax.set_title("Random Forest — AUC vs. Number of Trees", fontweight="bold")
plt.tight_layout()
plt.show()

AUC stabilizes around 200 trees, with negligible gains beyond that point. An n_estimators value of 500 was chosen to capture residual value while reducing compute time.

### 5.2 Hyperparameter Tuning

In [ ]:
pipe_rf = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

# n_estimators fixed at 500 per learning curve above; stabilization
# threshold identified empirically. Grid search tunes max_depth and
# min_samples_leaf only.
#
# max_depth: None allows full tree growth (unconstrained); 10 and 20 test
# progressively constrained trees to evaluate whether limiting depth
# reduces overfitting without sacrificing discrimination.
#
# min_samples_leaf: controls minimum node size. 1 allows single-sample
# leaves (low bias, high variance); 5 and 10 increase regularization
# by requiring larger leaf populations before a split is accepted.
param_grid_rf = {
    "classifier__n_estimators": [500],
    "classifier__max_depth": [None, 10, 20],
    "classifier__min_samples_leaf": [1, 5, 10],
}

grid_rf = GridSearchCV(
    pipe_rf,
    param_grid_rf,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1,
)
grid_rf.fit(X_train, y_train)

print(f"Best params (RF) : {grid_rf.best_params_}")
print(f"Best CV AUC (RF) : {grid_rf.best_score_:.4f}")

### 5.3 Test Set Evaluation

In [ ]:
best_rf = grid_rf.best_estimator_

y_pred_rf = best_rf.predict(X_test)
y_prob_rf = best_rf.predict_proba(X_test)[:, 1]

metrics_rf = {
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_test, y_pred_rf),
    "Precision": precision_score(y_test, y_pred_rf),
    "Recall": recall_score(y_test, y_pred_rf),
    "F1": f1_score(y_test, y_pred_rf),
    "ROC AUC": roc_auc_score(y_test, y_prob_rf),
}

print("Random Forest: Test Set Performance")
for k, v in metrics_rf.items():
    if k != "Model":
        print(f"  {k:<12} {v:.4f}")
print(f"\n  SUPPORT benchmark AUC: {SUPPORT_AUC:.4f}")

### 5.4 Feature Importances

In [ ]:
# Feature importances — mean decrease in impurity across all trees.
# Provides a model-native interpretability proxy; compare top features
# against LASSO coefficients and EDA hypothesis testing rankings.
rf_clf = best_rf.named_steps["classifier"]
importance_df = pd.DataFrame(
    {
        "Feature": preprocessed_feature_names,
        "Importance": rf_clf.feature_importances_,
    }
).sort_values("Importance", ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
top20 = importance_df.head(20)
ax.barh(top20["Feature"][::-1], top20["Importance"][::-1], edgecolor="white")
ax.set_xlabel("Mean Decrease in Impurity")
ax.set_title("Random Forest — Top 20 Feature Importances", fontweight="bold")
plt.tight_layout()
plt.show()

print("\nTop 15 features by importance:")
print(importance_df.head(15).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_rf),
    display_labels=["Survived", "Died"],
).plot(ax=ax, colorbar=False)
ax.set_title("Confusion Matrix — Random Forest", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Section 6: ROC Curve Comparison

All three candidate models are evaluated against each other and the SUPPORT 
benchmark using ROC curves and a summary metrics table. AUC is the primary 
comparison metric as it is threshold-agnostic and directly comparable to the 
0.787 benchmark reported by Knaus et al. (1995).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

models = [
    ("Logistic Regression (baseline)", y_prob_lr, "steelblue", "-"),
    ("LASSO Logistic Regression", y_prob_lasso, "darkorange", "-"),
    ("Random Forest", y_prob_rf, "seagreen", "-"),
]

for name, probs, color, ls in models:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    ax.plot(
        fpr,
        tpr,
        color=color,
        linestyle=ls,
        linewidth=2,
        label=f"{name} (AUC = {auc:.4f})",
    )

# SUPPORT benchmark — horizontal reference at the reported AUC value
ax.axhline(
    SUPPORT_AUC,
    color="crimson",
    linestyle=":",
    linewidth=1.5,
    label=f"SUPPORT benchmark — Knaus et al. (1995) AUC = {SUPPORT_AUC:.4f}",
)
ax.plot(
    [0, 1],
    [0, 1],
    color="gray",
    linestyle="--",
    linewidth=1,
    label="Random classifier (AUC = 0.50)",
)

ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title(
    "ROC Curve Comparison — All Models vs. SUPPORT Benchmark",
    fontsize=12,
    fontweight="bold",
)
ax.legend(fontsize=9, loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side metrics table with delta vs. SUPPORT benchmark
all_metrics = pd.DataFrame([metrics_lr, metrics_lasso, metrics_rf]).set_index("Model")
all_metrics["vs. SUPPORT AUC"] = (all_metrics["ROC AUC"] - SUPPORT_AUC).round(4)

print("Model Comparison Summary:\n")
print(all_metrics.round(4).to_string())
print(f"\nSUPPORT benchmark AUC: {SUPPORT_AUC}  (Knaus et al., 1995)")

---
## Section 7: Model Selection & Justification

Three candidate models were trained and evaluated against the SUPPORT prognostic
model benchmark (AUC = 0.787; Knaus et al., 1995).

| Model | Accuracy | Precision | Recall | F1 | ROC AUC | vs. SUPPORT |
|---|---|---|---|---|---|---|
| Logistic Regression (baseline) | 0.8096 | 0.7920 | 0.8024 | 0.7972 | 0.8852 | +0.098 |
| LASSO Logistic Regression | 0.8096 | 0.7920 | 0.8024 | 0.7972 | 0.8851 | +0.098 |
| Random Forest | 0.8195 | 0.8106 | 0.8000 | 0.8053 | 0.8959 | +0.109 |

### SUPPORT Model Comparison

The SUPPORT prognostic model (Knaus et al., 1995) was a Cox proportional hazards
model built on five components: primary disease category, a composite physiology
score (sps) derived from 11 day-3 lab variables, neurologic function (Glasgow
coma scale), patient age, and number of prior hospital days. It was developed on
4,301 phase I patients and validated on 4,028 independent phase II patients,
achieving an AUC of 0.78 on the phase II holdout.

Our EDA [EDA REF] confirmed that the SUPPORT model's surv6m survival probability
estimates achieve an AUC of 0.787 on the SUPPORT2 dataset (the random sample of
phases I and II used in this project) establishing a directly comparable benchmark
on the same data our models were trained and tested on.

Our Random Forest model achieves an AUC of 0.8959 on the same held-out test set,
exceeding the SUPPORT benchmark by 0.109. This improvement is achieved under more
constrained conditions than the original SUPPORT model:

- `sps` (SUPPORT physiology sub-score) excluded: a direct input to the SUPPORT
  model, excluded here to avoid circular prediction
- `aps` (APACHE III score) excluded: Knaus et al. evaluated SUPPORT and APACHE III
  as independent competing systems; including either as a feature in the other has
  no methodological precedent
- `prg6m`, `prg2m` (physician estimates) excluded: removing the physician-enhanced
  component that Knaus et al. showed improved SUPPORT AUC to 0.82

The comparison is not perfectly controlled; the SUPPORT model was a survival model
predicting continuous time-to-event outcomes while our Random Forest is a binary
classifier predicting 180-day mortality. AUC measures discrimination in both cases
but the underlying modeling frameworks differ. This limitation should be acknowledged
in the technical report.

### Selected model: Random Forest

All three models substantially exceeded the SUPPORT benchmark. The AUC difference
between Random Forest (0.8959) and the logistic models (0.8851–0.8852) is 0.011,
which is within normal sampling variation on a single holdout set and not
statistically significant on its own. Model selection therefore rests on
scientific grounds rather than raw AUC.

The project's primary objective is identifying which day-3 clinical and demographic
factors most strongly inform 180-day mortality. This was addressed in the EDA phase
through Mann-Whitney U hypothesis testing [EDA REF], interaction analysis
[EDA REF], and disease group stratification [EDA REF]. With interpretability
handled upstream, the modeling phase is free to optimize for predictive performance.

Random Forest is selected on three grounds:

1. **Highest AUC**: 0.8959 vs. 0.8851 for LASSO, representing the strongest
   discrimination between survivors and non-survivors on held-out data.

2. **Interaction capture**: Random Forest implicitly models the albumin × disease
   group and age × disease group interactions identified in EDA Section 8 without
   requiring explicit interaction terms. These interactions were a defining feature
   of the original SUPPORT model (Knaus et al., 1995) and are biologically
   meaningful.

3. **LASSO redundancy**: Baseline logistic regression and LASSO produced identical
   metrics on every measure. When regularization adds nothing, the feature set is
   already clean, validating EDA feature selection. However, it also means
   logistic regression offers no interpretability advantage over the baseline;
   there is no logistic model to prefer.

The 0.011 AUC gap over the SUPPORT benchmark's 30-year-old prognostic system,
achieved using raw clinical measurements alone and without the SUPPORT physiology
sub-score (sps) or APACHE III score (aps), demonstrates that modern ensemble
methods can match or exceed expert-designed scoring systems on this population.

In [ ]:
FINAL_MODEL = best_rf
FINAL_MODEL_NAME = "Random Forest"

y_pred_final = FINAL_MODEL.predict(X_test)
y_prob_final = FINAL_MODEL.predict_proba(X_test)[:, 1]

print(f"Final Model: {FINAL_MODEL_NAME}")
print(f"  Accuracy    {accuracy_score(y_test, y_pred_final):.4f}")
print(f"  Precision   {precision_score(y_test, y_pred_final):.4f}")
print(f"  Recall      {recall_score(y_test, y_pred_final):.4f}")
print(f"  F1          {f1_score(y_test, y_pred_final):.4f}")
print(f"  ROC AUC     {roc_auc_score(y_test, y_prob_final):.4f}")
print(f"\nSUPPORT benchmark AUC : {SUPPORT_AUC}")